In [7]:
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

import os

relative_path = "pose_landmarker_full.task"
model_path = os.path.abspath(relative_path)

print(model_path)

c:\Users\vanam\z\_winter\agent\backend_cv\experiments\pose_landmarker_full.task


In [8]:
BaseOptions = mp.tasks.BaseOptions
PoseLandmarker = mp.tasks.vision.PoseLandmarker
PoseLandmarkerOptions = mp.tasks.vision.PoseLandmarkerOptions
VisionRunningMode = mp.tasks.vision.RunningMode

options = PoseLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=model_path),
    running_mode=VisionRunningMode.IMAGE)

with PoseLandmarker.create_from_options(options) as landmarker:
    pass

In [9]:
def log_landmarks(result):
    print(result)

In [10]:
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

def plot_3d(landmarks):
    fig = plt.figure()
    ax = fig.add_subplot(111, projection='3d')

    xs = [lm.x for lm in landmarks]
    ys = [lm.y for lm in landmarks]
    zs = [lm.z for lm in landmarks]

    ax.scatter(xs, ys, zs)

    ax.set_xlabel("X")
    ax.set_ylabel("Y")
    ax.set_zlabel("Z")

    plt.show()

In [ ]:
import cv2
import mediapipe as mp
from mediapipe.tasks.python import vision
import os

# Путь к модели
relative_path = "pose_landmarker_full.task"
model_path = os.path.abspath(relative_path)
print("Using model:", model_path)

# Импорты Tasks API
PoseLandmarker = vision.PoseLandmarker
PoseLandmarkerOptions = vision.PoseLandmarkerOptions
VisionRunningMode = vision.RunningMode

BaseOptions = mp.tasks.BaseOptions
# Настройки landmarker
options = PoseLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=model_path),
    running_mode=VisionRunningMode.VIDEO  # поток с камеры
)

# Создаём landmarker
landmarker = PoseLandmarker.create_from_options(options)

# Открываем камеру
cap = cv2.VideoCapture(0)
cap.set(3, 640)
cap.set(4, 480)

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # Конвертируем кадр в RGB (MediaPipe требует)
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    # MediaPipe кадр
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)

    # detect_for_video требует timestamp (ms)
    timestamp = int(cap.get(cv2.CAP_PROP_POS_MSEC))
    result = landmarker.detect_for_video(mp_image, timestamp)

    # Рисуем landmark'ы на кадре
    log_landmarks(result.pose_landmarks)
    if result.pose_landmarks:
        for pose in result.pose_landmarks:
            plot_3d(pose)
            for lm in pose:
                x = int(lm.x * frame.shape[1])
                y = int(lm.y * frame.shape[0])
                cv2.circle(frame, (x, y), 5, (0, 255, 0), -1)

    # Показываем
    cv2.imshow("Pose Tracking", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
landmarker.close()


Using model: c:\Users\vanam\z\_winter\agent\backend_cv\experiments\pose_landmarker_full.task


In [13]:
import cv2
import mediapipe as mp
from mediapipe.tasks.python import vision
BaseOptions = mp.tasks.BaseOptions

# Путь к скачанной модели
MODEL_PATH = "pose_landmarker_full.task"  # обязательно правильный .task файл

# Конфигурация PoseLandmarker
options = vision.PoseLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=MODEL_PATH),
    running_mode=vision.RunningMode.VIDEO,
)

# Создаём landmarker
landmarker = vision.PoseLandmarker.create_from_options(options)

cap = cv2.VideoCapture(0)
cap.set(3, 640)
cap.set(4, 480)

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # Конвертируем для задачи
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))

    # Обработка кадра
    result = landmarker.detect_for_video(mp_image, int(cap.get(cv2.CAP_PROP_POS_MSEC)))

    # Рисуем landmark‑ы
    if result.pose_landmarks:
        for pose in result.pose_landmarks:
            for lm in pose:
                x = int(lm.x * frame.shape[1])
                y = int(lm.y * frame.shape[0])
                cv2.circle(frame, (x, y), 4, (0, 255, 0), -1)

    cv2.imshow("Pose Tracking", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

In [ ]:
import cv2
import mediapipe as mp
import imageio

from mediapipe.tasks.python import vision
BaseOptions = mp.tasks.BaseOptions

MODEL_PATH = "pose_landmarker_full.task"

options = vision.PoseLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=MODEL_PATH),
    running_mode=vision.RunningMode.VIDEO,
)

landmarker = vision.PoseLandmarker.create_from_options(options)

# Читаем гифку как список кадров
gif = imageio.mimread("input.gif")

# Размер, к которому хотим привести
TARGET_SIZE = (640, 480)

timestamp = 0
frame_time = 33  # ~30 FPS

while True:
    for frame in gif:
        # imageio -> numpy (RGB)
        frame = cv2.cvtColor(frame, cv2.COLOR_RGB2BGR)

        # 🔥 ресайз
        frame = cv2.resize(frame, TARGET_SIZE)

        # MediaPipe требует RGB
        mp_image = mp.Image(
            image_format=mp.ImageFormat.SRGB,
            data=cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        )

        result = landmarker.detect_for_video(mp_image, timestamp)
        timestamp += frame_time

        # Рисуем точки
        if result.pose_landmarks:
            for pose in result.pose_landmarks:
                for lm in pose:
                    x = int(lm.x * frame.shape[1])
                    y = int(lm.y * frame.shape[0])
                    cv2.circle(frame, (x, y), 4, (0, 255, 0), -1)

        cv2.imshow("Pose Tracking (GIF)", frame)

        if cv2.waitKey(frame_time) & 0xFF == ord('q'):
            exit()

KeyboardInterrupt: 

In [4]:
import cv2
import mediapipe as mp
import imageio

from mediapipe.tasks.python import vision
BaseOptions = mp.tasks.BaseOptions

MODEL_PATH = "pose_landmarker_full.task"

options = vision.PoseLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=MODEL_PATH),
    running_mode=vision.RunningMode.VIDEO,
)

landmarker = vision.PoseLandmarker.create_from_options(options)

gif = imageio.mimread("input.gif")

TARGET_SIZE = (640, 480)

timestamp = 0
frame_time = 33  # ~30 FPS

output_frames = []  # 🔥 сюда будем складывать результат

for frame in gif:
    frame = cv2.cvtColor(frame, cv2.COLOR_RGB2BGR)
    frame = cv2.resize(frame, TARGET_SIZE)

    mp_image = mp.Image(
        image_format=mp.ImageFormat.SRGB,
        data=cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    )

    result = landmarker.detect_for_video(mp_image, timestamp)
    timestamp += frame_time

    if result.pose_landmarks:
        for pose in result.pose_landmarks:
            for lm in pose:
                x = int(lm.x * frame.shape[1])
                y = int(lm.y * frame.shape[0])
                cv2.circle(frame, (x, y), 4, (0, 255, 0), -1)

    cv2.imshow("Pose Tracking (GIF)", frame)

    # 🔥 сохраняем кадр (нужно обратно в RGB!)
    output_frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))

    if cv2.waitKey(frame_time) & 0xFF == ord('q'):
        break

cv2.destroyAllWindows()

# 🔥 сохраняем гифку
imageio.mimsave("output.gif", output_frames, duration=frame_time / 1000)

In [10]:
import numpy as np

def angle(a, b, c):
    a, b, c = np.array(a), np.array(b), np.array(c)
    ba = a - b
    bc = c - b

    cos_angle = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc) + 1e-6)
    return np.degrees(np.arccos(cos_angle))

In [1]:
import numpy as np

def angle(a, b, c):
    a, b, c = np.array(a), np.array(b), np.array(c)
    ba = a - b
    bc = c - b

    cos_angle = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc) + 1e-6)
    return np.degrees(np.arccos(cos_angle))

def extract_angles(landmarks):
    # индексы mediapipe
    HIP = 23
    KNEE = 25
    ANKLE = 27
    SHOULDER = 11

    hip = landmarks[HIP]
    knee = landmarks[KNEE]
    ankle = landmarks[ANKLE]
    shoulder = landmarks[SHOULDER]

    knee_angle = angle(hip, knee, ankle)
    hip_angle = angle(shoulder, hip, knee)

    return [knee_angle, hip_angle]

def dtw(seq1, seq2):
    n, m = len(seq1), len(seq2)
    dp = np.full((n+1, m+1), np.inf)
    dp[0, 0] = 0

    for i in range(1, n+1):
        for j in range(1, m+1):
            cost = np.linalg.norm(np.array(seq1[i-1]) - np.array(seq2[j-1]))
            dp[i, j] = cost + min(
                dp[i-1, j],
                dp[i, j-1],
                dp[i-1, j-1]
            )
    return dp[n, m] / (n + m)

def process_gif(frames, landmarker):
    sequence = []
    frame_ids = []

    frames = imageio.mimread(frames)
    timestamp = 0

    for i, frame in enumerate(frames):
        mp_image = mp.Image(
            image_format=mp.ImageFormat.SRGB,
            data=frame
        )

        result = landmarker.detect_for_video(mp_image, timestamp)
        timestamp += 33

        if result.pose_landmarks:
            lm = result.pose_landmarks[0]
            coords = [(p.x, p.y) for p in lm]

            sequence.append(extract_angles(coords))
            frame_ids.append(i)  # 🔥 ВАЖНО

    return sequence, frame_ids


def create_landmarker():
    return vision.PoseLandmarker.create_from_options(options)

seq_ref, _ = process_gif('./input.gif', create_landmarker())
seq_user, user_ids = process_gif('./input2.gif', create_landmarker())

score = dtw(seq_ref, seq_user)

print("Distance:", score)

NameError: name 'vision' is not defined

In [31]:
def frame_errors(seq_ref, seq_user):
    min_len = min(len(seq_ref), len(seq_user))
    errors = []

    for i in range(min_len):
        err = np.linalg.norm(
            np.array(seq_ref[i]) - np.array(seq_user[i])
        )
        errors.append(err)

    return errors

def visualize_gif(path, landmarker, errors, frame_ids, threshold=20):
    frames = imageio.mimread(path)
    timestamp = 0

    # 🔥 делаем быстрый мап: frame_id → error
    error_map = {fid: err for fid, err in zip(frame_ids, errors)}

    while True:
        for i, frame in enumerate(frames):
            frame = cv2.cvtColor(frame, cv2.COLOR_RGB2BGR)

            mp_image = mp.Image(
                image_format=mp.ImageFormat.SRGB,
                data=cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            )

            result = landmarker.detect_for_video(mp_image, timestamp)
            timestamp += 33

            # 🔥 теперь правильно: смотрим по frame_ids
            if i in error_map:
                err = error_map[i]

                if err < threshold:
                    color = (0, 255, 0)
                    text = "OK"
                else:
                    color = (0, 0, 255)
                    text = "BAD"

                cv2.putText(frame, f"{text} {err:.1f}",
                            (20, 40),
                            cv2.FONT_HERSHEY_SIMPLEX,
                            1, color, 2)
            else:
                # если кадр без детекции
                cv2.putText(frame, "NO POSE",
                            (20, 40),
                            cv2.FONT_HERSHEY_SIMPLEX,
                            1, (0, 255, 255), 2)

            # landmarks
            if result.pose_landmarks:
                for lm in result.pose_landmarks[0]:
                    x = int(lm.x * frame.shape[1])
                    y = int(lm.y * frame.shape[0])
                    cv2.circle(frame, (x, y), 3, (255, 255, 255), -1)

            cv2.imshow("Result", frame)

        if cv2.waitKey(30) & 0xFF == ord('q'):
            break

    cv2.destroyAllWindows()

In [ ]:
seq_ref, _ = process_gif('./input.gif', create_landmarker())
seq_user, user_frame_ids = process_gif('./input2.gif', create_landmarker())

errors = frame_errors(seq_ref, seq_user)

visualize_gif('./input2.gif', create_landmarker(), errors, user_frame_ids)

Exception ignored in: <function Image.__del__ at 0x000001E48D8BF6A0>
Traceback (most recent call last):
  File "c:\Users\vanam\Z\_winter\agent\backend_cv\experiments\.venv\Lib\site-packages\mediapipe\tasks\python\vision\core\image.py", line 553, in __del__
    if self._image_ptr:
       ^^^^^^^^^^^^^^^
AttributeError: 'Image' object has no attribute '_image_ptr'
